In [7]:

import pandas as pd
mv=pd.read_excel(r"C:\Users\RohanS\Downloads\movie.xlsx")
rt=pd.read_csv(r"C:\Users\RohanS\Downloads\ratings.csv")

mv['genres']=(mv['genres'].str.replace('|',' ',regex=False))
mv['title'] = mv['title'].str.strip()


In [ ]:

movie_counts = rt.groupby(
    'movieId'
).size()

popular_movies = movie_counts[
    movie_counts >= 0
].index
filtered_ratings = rt[
    rt['movieId'].isin(popular_movies)
]
movie_user = filtered_ratings.pivot_table(
    index='movieId',
    columns='userId',
    values='rating'
)
from scipy.sparse import csr_matrix
import numpy as np

movie_counts = rt.groupby(
    'movieId'
).size()

popular_movies = movie_counts[
    movie_counts >= 10000
].index

filtered_ratings = rt[
    rt['movieId'].isin(popular_movies)
]


C:\Users\RohanS\AppData\Local\Temp\ipykernel_8592\2357271074.py:11: PerformanceWarning: The following operation may generate 3703856792 cells in the resulting pandas object.
  movie_user = filtered_ratings.pivot_table(


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

movie_user_filled = movie_user.fillna(0)
movie_similarity = cosine_similarity(movie_user_filled)

In [ ]:
movie_similarity_df = pd.DataFrame(
    movie_similarity,
    index=movie_user.index,
    columns=movie_user.index
)

In [ ]:
movie_lookup = mv[
    ['movieId','title']
]

In [ ]:
def recommend_movie(movie_title, n=10):

    movie_id = movie_lookup[
        movie_lookup['title'] == movie_title
    ]['movieId'].iloc[0]

    similar_movies = (
        movie_similarity_df[movie_id]
        .sort_values(ascending=False)
        .iloc[1:n+1]
    )

    recommendations = pd.DataFrame({
        'movieId': similar_movies.index,
        'similarity': similar_movies.values
    })

    recommendations = recommendations.merge(
        movie_lookup,
        on='movieId'
    )

    return recommendations[
        ['title','similarity']
    ]
recommend_movie("Interstellar")